In [2]:
import pandas as pd
import numpy as np
import os

In [5]:
df = pd.read_csv('../data/raw/raw_data.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
df.head()

Shape: (258, 11)

Columns: ['timestamp', 'age', 'year', 'major', 'gender', 'food', 'transport', 'entertainment', 'personal_care', 'misc', 'allowance']

Dtypes:
timestamp          str
age                str
year               str
major              str
gender             str
food             int64
transport        int64
entertainment    int64
personal_care    int64
misc             int64
allowance        int64
dtype: object

First 5 rows:


,timestamp,age,year,major,gender,food,transport,entertainment,personal_care,misc,allowance
0,4/20/2026 23:14:12,21,Junior,MGS,Male,20000,4000,5000,20000,2000,20000
1,4/20/2026 23:14:39,20,Sophomore,FCSE,Female,18000,6000,0,0,1000,25000
2,4/20/2026 23:15:59,19,Sophomore,FCSE,Male,15000,0,3000,0,2000,25000
3,4/20/2026 23:16:59,20,Freshman,Engineering,Male,26000,0,2000,2000,2000,30000
4,4/20/2026 23:19:29,21,Sophomore,FCSE,Female,25000,8000,5000,0,0,40000


In [6]:
df = df.drop(columns=['timestamp'])

print(f"Shape after dropping timestamp: {df.shape}")

Shape after dropping timestamp: (258, 10)


In [7]:
numeric_cols = ['age', 'food', 'transport', 'entertainment', 'personal_care', 'misc', 'allowance']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("NaN counts per column after coercion:")
print(df.isnull().sum())

NaN counts per column after coercion:
age              3
year             0
major            0
gender           0
food             0
transport        0
entertainment    0
personal_care    0
misc             0
allowance        0
dtype: int64


In [8]:
dirty_rows = df[df.isnull().any(axis=1)]
print(f"Rows with at least one NaN: {len(dirty_rows)}")
print(dirty_rows.to_string())

Rows with at least one NaN: 3
     age       year        major  gender   food  transport  entertainment  personal_care  misc  allowance
53   NaN  Sophomore  Engineering    Male  10000       3000           5000           1500     0      16000
65   NaN     Junior  Engineering    Male  16000       4000           3500           5000   500      35000
117  NaN  Sophomore         FCSE  Female  28000       8000              0              0     0      29000


In [9]:
initial_count = len(df)

# Drop rows where age is NaN or out of realistic range
df = df[df['age'].notna()]
df = df[(df['age'] >= 16) & (df['age'] <= 30)]

# Drop rows where allowance is NaN, too low, or absurdly high
df = df[df['allowance'].notna()]
df = df[(df['allowance'] >= 500) & (df['allowance'] <= 500_000)]

# Drop rows where any spending column is NaN
spending_cols = ['food', 'transport', 'entertainment', 'personal_care', 'misc']
df = df.dropna(subset=spending_cols)

dropped = initial_count - len(df)
print(f"Started with: {initial_count} rows")
print(f"Dropped:      {dropped} rows")
print(f"Remaining:    {len(df)} rows")

Started with: 258 rows
Dropped:      14 rows
Remaining:    244 rows


In [10]:
df['total_spend'] = df[spending_cols].sum(axis=1)

mean_spend = df['total_spend'].mean()
std_spend  = df['total_spend'].std()

upper_bound = mean_spend + 3 * std_spend

before = len(df)
df = df[df['total_spend'] <= upper_bound]
after = len(df)

print(f"Spend mean:   PKR {mean_spend:,.0f}")
print(f"Spend std:    PKR {std_spend:,.0f}")
print(f"Upper bound:  PKR {upper_bound:,.0f}")
print(f"Rows removed: {before - after}")
print(f"Remaining:    {after}")

Spend mean:   PKR 39,268
Spend std:    PKR 35,929
Upper bound:  PKR 147,055
Rows removed: 4
Remaining:    240


In [11]:
df = df.reset_index(drop=True)

print(f"Final clean dataset: {df.shape}")
print(f"\nIndex now runs from 0 to {len(df)-1}")

Final clean dataset: (240, 11)

Index now runs from 0 to 239


In [12]:
# Target variable: sum of all 5 spending categories
df['total_spend'] = df[['food', 'transport', 'entertainment', 'personal_care', 'misc']].sum(axis=1)

# Surplus: positive means money left over, negative means in deficit
df['surplus'] = df['allowance'] - df['total_spend']

# Financial health classification
def classify(row):
    buffer = 0.05 * row['allowance']
    if row['surplus'] > buffer:
        return 'Saving'
    elif row['surplus'] < -buffer:
        return 'Overspending'
    else:
        return 'Balanced'

df['financial_status'] = df.apply(classify, axis=1)

print("Financial status distribution:")
print(df['financial_status'].value_counts())
print(f"\nMedian total spend:  PKR {df['total_spend'].median():,.0f}")
print(f"Median allowance:    PKR {df['allowance'].median():,.0f}")
print(f"Median surplus:      PKR {df['surplus'].median():,.0f}")

Financial status distribution:
financial_status
Overspending    115
Saving           94
Balanced         31
Name: count, dtype: int64

Median total spend:  PKR 32,000
Median allowance:    PKR 31,000
Median surplus:      PKR -1,000


In [13]:
print("=== FINAL DATASET SUMMARY ===")
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nNull check:\n{df.isnull().sum()}")
print(f"\nSample:\n")
df.head(10)

=== FINAL DATASET SUMMARY ===
Rows: 240
Columns: ['age', 'year', 'major', 'gender', 'food', 'transport', 'entertainment', 'personal_care', 'misc', 'allowance', 'total_spend', 'surplus', 'financial_status']

Dtypes:
age                 float64
year                    str
major                   str
gender                  str
food                  int64
transport             int64
entertainment         int64
personal_care         int64
misc                  int64
allowance             int64
total_spend           int64
surplus               int64
financial_status        str
dtype: object

Null check:
age                 0
year                0
major               0
gender              0
food                0
transport           0
entertainment       0
personal_care       0
misc                0
allowance           0
total_spend         0
surplus             0
financial_status    0
dtype: int64

Sample:



,age,year,major,gender,food,transport,entertainment,personal_care,misc,allowance,total_spend,surplus,financial_status
0,21.0,Junior,MGS,Male,20000,4000,5000,20000,2000,20000,51000,-31000,Overspending
1,20.0,Sophomore,FCSE,Female,18000,6000,0,0,1000,25000,25000,0,Balanced
2,19.0,Sophomore,FCSE,Male,15000,0,3000,0,2000,25000,20000,5000,Saving
3,20.0,Freshman,Engineering,Male,26000,0,2000,2000,2000,30000,32000,-2000,Overspending
4,21.0,Sophomore,FCSE,Female,25000,8000,5000,0,0,40000,38000,2000,Balanced
5,19.0,Sophomore,FCSE,Male,45000,28000,3000,3000,2000,78000,81000,-3000,Balanced
6,20.0,Sophomore,FCSE,Male,32000,0,4000,1500,3000,40000,40500,-500,Balanced
7,20.0,Sophomore,FCSE,Male,30000,1000,5000,5000,3000,40000,44000,-4000,Overspending
8,22.0,Junior,MGS,Male,15000,4000,2500,2000,1000,20000,24500,-4500,Overspending
9,19.0,Freshman,MGS,Male,40000,12000,20000,0,0,50000,72000,-22000,Overspending


In [14]:
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/clean_survey.csv', index=False)

print("Saved to ../data/processed/clean_survey.csv")
print(f"Final shape: {df.shape}")

Saved to ../data/processed/clean_survey.csv
Final shape: (240, 13)
